# Lecture 07: Other Supervised Learning Approaches and Evaluation — Exercises

Companion to the note [Noter små](../Noter%20små.md).

Practise naive Bayes (the Bayesian network with the simplest structure from the note), k-nearest neighbours with hand-made distance functions, overfitting and post-pruning, and the evaluation toolbox: confusion matrices, precision/recall, ROC curves and cross-validation. By hand first, then from scratch in NumPy with sklearn as reference.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import pandas as pd
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris, load_breast_cancer, make_moons
rng = np.random.default_rng(7)
# Toy spam data: features (free, money, meeting), label spam (1) / not spam (0)
X_mail = np.array([[1, 1, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0],
                   [0, 0, 1], [1, 0, 1], [0, 0, 0], [0, 1, 1]])
y_mail = np.array([1, 1, 1, 1, 0, 0, 0, 0])
# Loan data for k-NN: Income (0=low,1=medium,2=high), Houseowner (1=yes), Marital (0=unmarried,1=married,2=divorced)
X_loan = np.array([[2, 1, 1], [1, 1, 0], [1, 0, 2], [0, 1, 1], [0, 0, 0], [2, 1, 0]])
y_loan = np.array(['yes', 'yes', 'no', 'no', 'no', 'yes'])
x_new_loan = np.array([2, 0, 2])   # high income, no house, divorced
def loan_dist(a, b):
    return abs(a[0] - b[0]) + (a[1] != b[1]) + (a[2] != b[2])
# A learned decision tree (leaves are labels) and a validation set for post-pruning
tree_full = {'Hot': {'y': {'Fly': {'y': 'yes', 'n': {'Nature': {'y': 'yes', 'n': 'no'}}}},
                     'n': {'Nature': {'y': 'yes', 'n': {'Fly': {'y': 'yes', 'n': 'no'}}}}}}
val = pd.DataFrame({'Fly':    ['n', 'n', 'y', 'n', 'n'],
                    'Hot':    ['y', 'y', 'n', 'y', 'n'],
                    'Nature': ['y', 'n', 'n', 'y', 'n'],
                    'Likes':  ['no', 'yes', 'yes', 'yes', 'no']})
def tree_predict(tree, row):
    while isinstance(tree, dict):
        a = next(iter(tree)); tree = tree[a][row[a]]
    return tree

## Part A · Concepts

### Exercise 1 · The naive Bayes assumption
*🧠 Concept · ★☆☆*

The note models spam with $P(a_1,\dots,a_n,\text{Spam}) = P(\text{Spam})\prod_i P(a_i\mid\text{Spam})$.

1. Draw the Bayesian network. Which conditional independence statement does it encode?
2. Give an example of two words for which the assumption is clearly false.
3. Why does naive Bayes often classify well even when the assumption is false?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Spam is the only parent of every word node.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Spam → $a_1$, Spam → $a_2$, …, Spam → $a_n$. It says that the words are **conditionally independent given the class**:
   $P(a_i\mid a_j,\text{Spam}) = P(a_i\mid\text{Spam})$.
2. "Hong" and "Kong", or "viagra" and "pills": given that an email is spam, seeing one makes the other far more likely.
3. Classification only needs the **argmax** of the posterior to be right, not calibrated probabilities. Correlated words make naive Bayes
   over-confident (evidence is counted twice), but they often push in the right direction. And with few parameters it needs little data.

</details>

### Exercise 2 · Overfitting and Ockham's razor
*🧠 Concept · ★☆☆*

Two hypotheses on the same task:

| | training error | error on new data |
|:--:|:--:|:--:|
| $h$ (deep tree) | 0.02 | 0.20 |
| $h'$ (shallow tree) | 0.10 | 0.12 |

1. Using the note's definition, which hypothesis overfits?
2. A linear function and a degree-9 polynomial both pass close to 10 measured points; the polynomial passes exactly through all of them.
   What does Ockham's razor say, and what goes wrong with the polynomial?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$h$ overfits if some $h'$ has larger training error but smaller error on the whole distribution.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $h$ overfits: $h'$ has larger training error (0.10 > 0.02) but smaller error over the distribution (0.12 < 0.20).
2. Ockham's razor prefers the **simplest hypothesis consistent with the data**: the line. The polynomial also fits the noise; between and
   beyond the measured points it oscillates wildly, so predictions for new measurements are poor (high variance).

</details>

### Exercise 3 · Accuracy can lie
*🧠 Concept · ★★☆*

Only 5 % of emails are spam. A "classifier" that always answers *not spam* is evaluated on 1000 emails.

1. Compute its accuracy, recall and precision (spam = positive class).
2. Why is accuracy a poor metric here, and which metrics would you report instead?
3. The note says maximising the true-positive rate and minimising the false-positive rate are incompatible goals. Explain using a decision threshold.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

There are 50 spam and 950 non-spam emails; the classifier never predicts positive.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Accuracy $= 950/1000 = 0.95$; recall $= 0/50 = 0$; precision $= 0/0$ is undefined (often reported as 0).
2. With imbalanced classes a useless classifier gets high accuracy. Report recall and precision (or F1), the full confusion matrix, or ROC/AUC.
3. A probabilistic classifier says "spam" if $P(\text{spam}\mid x) > \tau$. Lowering $\tau$ catches more spam (TPR ↑) but also flags more
   normal mail (FPR ↑); raising it does the opposite. The ROC curve shows this trade-off over all thresholds.

</details>

### Exercise 4 · Choosing k in k-NN
*🧠 Concept · ★★☆*

1. What does 1-NN achieve on its own training data, and why is that number meaningless?
2. Describe how the decision boundary changes from $k = 1$ to $k = N$ (all training examples). Which end overfits, which underfits?
3. Why is an odd $k$ convenient for two classes?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

With $k = N$ every query gets the same neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. 100 % training accuracy (each point is its own nearest neighbour, assuming no duplicate inputs with different labels). It says nothing about
   generalisation; evaluate on held-out data or with cross-validation.
2. $k = 1$: a very jagged boundary that follows every noisy point (high variance, overfitting). Larger $k$ averages over more neighbours and smooths
   the boundary. $k = N$ always predicts the majority class (high bias, underfitting). The best $k$ lies in between and is chosen by validation.
3. It avoids ties in the majority vote.

</details>

## Part B · By hand

### Exercise 5 · Naive Bayes spam classification by hand
*✍️ By hand · ★★☆*

Training data (1 = word present):

| # | free | money | meeting | Spam |
|:--:|:--:|:--:|:--:|:--:|
| 1 | 1 | 1 | 0 | yes |
| 2 | 1 | 0 | 0 | yes |
| 3 | 0 | 1 | 0 | yes |
| 4 | 1 | 1 | 0 | yes |
| 5 | 0 | 0 | 1 | no |
| 6 | 1 | 0 | 1 | no |
| 7 | 0 | 0 | 0 | no |
| 8 | 0 | 1 | 1 | no |

Estimate the parameters by frequency counting and compute $P(\text{Spam}=\text{yes}\mid \text{free}=1,\text{money}=1,\text{meeting}=0)$.
Then compute it for an email with all three words present. What goes wrong?

In [ ]:
p_spam_q1 = None   # free=1, money=1, meeting=0
p_spam_q2 = None   # free=1, money=1, meeting=1

_m = BernoulliNB(alpha=1e-10, force_alpha=True).fit(X_mail, y_mail)
_r = _m.predict_proba([[1, 1, 0], [1, 1, 1]])[:, 1]
check('P(spam | 1,1,0)', p_spam_q1, _r[0], tol=1e-3); check('P(spam | 1,1,1)', p_spam_q2, _r[1], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$P(\text{free}\mid\text{yes}) = 3/4$, $P(\text{free}\mid\text{no}) = 1/4$, and the same for money. Count meeting yourself.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Unnormalised: $P(\text{yes})\prod_i P(a_i\mid\text{yes})$ versus $P(\text{no})\prod_i P(a_i\mid\text{no})$, then normalise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(\text{yes}) = P(\text{no}) = 1/2$. Spam: free $3/4$, money $3/4$, meeting $0/4$. Not spam: free $1/4$, money $1/4$, meeting $3/4$.

Query 1: yes: $\tfrac12\cdot\tfrac34\cdot\tfrac34\cdot1 = 0.28125$; no: $\tfrac12\cdot\tfrac14\cdot\tfrac14\cdot\tfrac14 = 0.0078$.
$P(\text{yes}\mid x) = 0.28125/0.28906 = 36/37 \approx 0.973$.

Query 2: $P(\text{meeting}\mid\text{yes}) = 0$ kills the whole product, so $P(\text{yes}\mid x) = 0$, although "free" and "money" strongly
indicate spam. One unseen word–class combination overrules all other evidence. Next exercise: smoothing.

```python
p_spam_q1 = 36 / 37
p_spam_q2 = 0.0
```

</details>

### Exercise 6 · Laplace smoothing
*✍️ By hand · ★★☆*

Redo both queries with Laplace (add-one) smoothing: $P(a_i = 1\mid c) = \frac{N(a_i=1, c) + 1}{N(c) + 2}$ (the class prior stays $1/2$).

In [ ]:
p_spam_q1_lap = None
p_spam_q2_lap = None

_m = BernoulliNB(alpha=1.0).fit(X_mail, y_mail)
_r = _m.predict_proba([[1, 1, 0], [1, 1, 1]])[:, 1]
check('smoothed P(spam | 1,1,0)', p_spam_q1_lap, _r[0], tol=1e-3); check('smoothed P(spam | 1,1,1)', p_spam_q2_lap, _r[1], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Spam: free and money become $4/6$, meeting becomes $1/6$. Not spam: free and money $2/6$, meeting $4/6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Query 1: yes $\propto \tfrac23\cdot\tfrac23\cdot\tfrac56 = 0.370$, no $\propto \tfrac13\cdot\tfrac13\cdot\tfrac13 = 0.037$; posterior $= 10/11 \approx 0.909$.

Query 2: yes $\propto \tfrac23\cdot\tfrac23\cdot\tfrac16 = 4/54$, no $\propto \tfrac13\cdot\tfrac13\cdot\tfrac46 = 4/54$; posterior $= 0.5$.
Now "meeting" is strong evidence against spam, but no longer infinitely strong. Smoothing acts like a prior of one imaginary example
of each value per class.

```python
p_spam_q1_lap = 10 / 11
p_spam_q2_lap = 0.5
```

</details>

### Exercise 7 · k-NN with a hand-made distance
*✍️ By hand · ★★☆*

Predict whether a person pays back a loan. The distance between two examples is the sum of per-feature distances:
Income $|{\rm low}=0, {\rm medium}=1, {\rm high}=2|$ difference, Houseowner 0 if equal else 1, Marital status 0 if equal else 1.

| # | Income | Houseowner | Marital | Pay back |
|:--:|:--:|:--:|:--:|:--:|
| 1 | high | yes | married | yes |
| 2 | medium | yes | unmarried | yes |
| 3 | medium | no | divorced | no |
| 4 | low | yes | married | no |
| 5 | low | no | unmarried | no |
| 6 | high | yes | unmarried | yes |

New case: Income = high, Houseowner = no, Marital = divorced. Compute all six distances and the 1-NN and 3-NN predictions.
Then suggest a change of the distance function that would flip the 1-NN prediction.

In [ ]:
dists = None    # list of 6 distances
pred_1nn = None
pred_3nn = None

_D = np.array([loan_dist(x_new_loan, x) for x in X_loan])
check('distances', dists, _D)
_k1 = KNeighborsClassifier(n_neighbors=1, metric='precomputed').fit(np.zeros((6, 6)), y_loan)
_k3 = KNeighborsClassifier(n_neighbors=3, metric='precomputed').fit(np.zeros((6, 6)), y_loan)
check('1-NN', pred_1nn, _k1.predict(_D[None, :])[0]); check('3-NN', pred_3nn, _k3.predict(_D[None, :])[0])

<details>
<summary><b>💡 Hint</b></summary>

Distances: example 1 is $0 + 1 + 1 = 2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Distances: $(2, 3, 1, 4, 3, 2)$. **1-NN**: example 3 (distance 1) → **no**. **3-NN**: examples 3, 1, 6 → no, yes, yes → **yes**.

To flip 1-NN, make income matter more, e.g. multiply the income difference by 3: example 3 moves to distance $3$ while examples 1 and 6
stay at $2$, so the nearest neighbours become the high-income people who paid back and 1-NN predicts **yes**. Choosing a
distance function is a modelling decision that encodes which features matter.

```python
dists = [2, 3, 1, 4, 3, 2]
pred_1nn = 'no'
pred_3nn = 'yes'
```

</details>

### Exercise 8 · Confusion-matrix metrics
*✍️ By hand · ★☆☆*

A spam detector on 1000 emails (spam = positive):

| | predicted spam | predicted not spam |
|:--:|:--:|:--:|
| **actual spam** | 40 | 10 |
| **actual not spam** | 20 | 930 |

Compute accuracy, recall, precision, specificity ($tn/(tn+fp)$), false positive rate and F1. Then recompute recall and precision with
*not spam* as the positive class.

In [ ]:
acc = None
rec = None
prec = None
spec = None
fpr = None
f1 = None
rec_swapped = None
prec_swapped = None

_y = np.r_[np.ones(50), np.zeros(950)]; _p = np.r_[np.ones(40), np.zeros(10), np.ones(20), np.zeros(930)]
check('accuracy', acc, accuracy_score(_y, _p), tol=1e-3); check('recall', rec, recall_score(_y, _p), tol=1e-3)
check('precision', prec, precision_score(_y, _p), tol=1e-3); check('specificity', spec, recall_score(_y, _p, pos_label=0), tol=1e-3)
check('FPR', fpr, 1 - recall_score(_y, _p, pos_label=0), tol=1e-3); check('F1', f1, f1_score(_y, _p), tol=1e-3)
check('recall (not spam positive)', rec_swapped, recall_score(_y, _p, pos_label=0), tol=1e-3)
check('precision (not spam positive)', prec_swapped, precision_score(_y, _p, pos_label=0), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$tp = 40$, $fn = 10$, $fp = 20$, $tn = 930$. F1 is the harmonic mean $2PR/(P+R)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Accuracy $= 970/1000 = 0.97$; recall $= 40/50 = 0.8$; precision $= 40/60 \approx 0.667$; specificity $= 930/950 \approx 0.979$;
FPR $= 20/950 \approx 0.021$; F1 $= 2\cdot0.667\cdot0.8/1.467 \approx 0.727$.

Swapping the classes: recall becomes $930/950 \approx 0.979$ (the old specificity) and precision $930/940 \approx 0.989$. Accuracy is unchanged.

```python
acc = 0.97
rec = 0.8
prec = 40 / 60
spec = 930 / 950
fpr = 20 / 950
f1 = 2 * (40 / 60) * 0.8 / (40 / 60 + 0.8)
rec_swapped = 930 / 950
prec_swapped = 930 / 940
```

</details>

### Exercise 9 · Moving the decision threshold
*✍️ By hand · ★★☆*

A classifier gives these spam scores to 8 test emails (1 = spam):

| score | 0.95 | 0.85 | 0.70 | 0.60 | 0.45 | 0.40 | 0.30 | 0.10 |
|:--:|:--:|:--:|:--:|:--:|:--:|:--:|:--:|:--:|
| label | 1 | 1 | 0 | 1 | 1 | 0 | 0 | 0 |

Compute TPR (recall), FPR and precision for threshold $\tau = 0.5$ and for $\tau = 0.35$ (predict spam if score $\ge \tau$).

In [ ]:
tpr_05 = None
fpr_05 = None
prec_05 = None
tpr_035 = None
fpr_035 = None
prec_035 = None

_s = np.array([0.95, 0.85, 0.70, 0.60, 0.45, 0.40, 0.30, 0.10]); _l = np.array([1, 1, 0, 1, 1, 0, 0, 0])
for _tau, _t, _f, _pr in [(0.5, tpr_05, fpr_05, prec_05), (0.35, tpr_035, fpr_035, prec_035)]:
    _p = (_s >= _tau).astype(int)
    check(f'TPR @ {_tau}', _t, recall_score(_l, _p), tol=1e-3)
    check(f'FPR @ {_tau}', _f, 1 - recall_score(_l, _p, pos_label=0), tol=1e-3)
    check(f'precision @ {_tau}', _pr, precision_score(_l, _p), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

There are 4 positives and 4 negatives. At $\tau = 0.5$ the top four emails are predicted spam.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\tau = 0.5$: predicted spam = top 4 (labels 1, 1, 0, 1): $tp = 3$, $fp = 1$. TPR $= 3/4 = 0.75$, FPR $= 1/4 = 0.25$, precision $= 0.75$.

$\tau = 0.35$: top 6 (adds labels 1, 0): $tp = 4$, $fp = 2$. TPR $= 1$, FPR $= 0.5$, precision $= 4/6 \approx 0.667$.

Lowering the threshold raised recall but also the false-positive rate, exactly the trade-off in the note. These are two points of the ROC curve.

```python
tpr_05 = 0.75
fpr_05 = 0.25
prec_05 = 0.75
tpr_035 = 1.0
fpr_035 = 0.5
prec_035 = 4 / 6
```

</details>

### Exercise 10 · Reduced-error post-pruning
*✍️ By hand · ★★☆*

A decision tree learned on 12 examples (100 % training accuracy):

```
Hot = y → Fly = y → yes
          Fly = n → Nature = y → yes (2 examples)
                    Nature = n → no  (1 example)
Hot = n → Nature = y → yes
          Nature = n → Fly = y → yes (1 example)
                       Fly = n → no  (2 examples)
```

Validation set:

| Fly | Hot | Nature | Likes |
|:--:|:--:|:--:|:--:|
| n | y | y | no |
| n | y | n | yes |
| y | n | n | yes |
| n | y | y | yes |
| n | n | n | no |

1. Compute the validation accuracy of the full tree.
2. Replace the *Nature* node under Hot = y, Fly = n by a leaf with its majority training label. New validation accuracy? Prune?
3. Next candidate: the *Fly* node under Hot = n, Nature = n (majority label *no*). Validation accuracy if it is also pruned? Prune?

In [ ]:
acc_full = None
acc_prune1 = None
prune1 = None   # True/False
acc_prune2 = None
prune2 = None

import copy
_acc = lambda t: np.mean([tree_predict(t, r) == r['Likes'] for _, r in val.iterrows()])
_t1 = copy.deepcopy(tree_full); _t1['Hot']['y']['Fly']['n'] = 'yes'
_t2 = copy.deepcopy(_t1); _t2['Hot']['n']['Nature']['n'] = 'no'
check('full tree', acc_full, _acc(tree_full)); check('after pruning Nature', acc_prune1, _acc(_t1))
check('prune Nature?', prune1, bool(_acc(_t1) >= _acc(tree_full)))
check('after also pruning Fly', acc_prune2, _acc(_t2)); check('prune Fly?', prune2, bool(_acc(_t2) >= _acc(_t1)))

<details>
<summary><b>💡 Hint</b></summary>

Prune a node if the validation accuracy does not get worse. The majority label of the first Nature node is *yes* (2 vs 1).

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Full tree predictions: (yes, no, yes, yes, no) versus actual (no, yes, yes, yes, no): rows 3, 4, 5 correct, accuracy **0.6**.
2. With the leaf *yes*: predictions (yes, yes, yes, yes, no): rows 2–5 correct, accuracy **0.8** ≥ 0.6, so **prune**.
3. With *no* under Hot=n, Nature=n: row 3 becomes wrong, accuracy **0.6** < 0.8, so **do not prune**.

The validation set has to be separate from the training data; otherwise pruning could never improve the (perfect) training accuracy.

```python
acc_full = 0.6
acc_prune1 = 0.8
prune1 = True
acc_prune2 = 0.6
prune2 = False
```

</details>

## Part C · Code

### Exercise 11 · Bernoulli naive Bayes from scratch
*💻 Code · ★★☆*

Implement `nb_fit(X, y, alpha)` returning class priors and the table $P(a_i = 1\mid c)$ with add-$\alpha$ smoothing, and
`nb_predict_proba(model, X)` returning $P(c\mid x)$ for each row. Work in **log space** to avoid underflow with many words.
Compare with `BernoulliNB(alpha=1.0)` on the random word data below.

In [ ]:
X_words = (rng.random((300, 50)) < 0.2).astype(int)
y_words = (X_words[:, :5].sum(1) + rng.random(300) > 1.5).astype(int)

def nb_fit(X, y, alpha=1.0):
    # TODO: return (classes, log_prior, theta) where theta[c, i] = P(a_i = 1 | c)
    return None

def nb_predict_proba(model, X):
    # TODO
    return None

nb_model = nb_fit(X_words, y_words)
proba_nb = None if nb_model is None else nb_predict_proba(nb_model, X_words)

check('matches BernoulliNB', proba_nb, BernoulliNB(alpha=1.0).fit(X_words, y_words).predict_proba(X_words))

<details>
<summary><b>💡 Hint 1</b></summary>

$\log P(c\mid x) = \log P(c) + \sum_i [x_i\log\theta_{ci} + (1 - x_i)\log(1 - \theta_{ci})] + \text{const}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Normalise with the log-sum-exp trick: subtract the row maximum before exponentiating.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Multiplying 50 probabilities of ~0.2 gives numbers around $10^{-35}$; with thousands of words this underflows to 0. Summing logs avoids it.

```python
X_words = (rng.random((300, 50)) < 0.2).astype(int)
y_words = (X_words[:, :5].sum(1) + rng.random(300) > 1.5).astype(int)

def nb_fit(X, y, alpha=1.0):
    classes = np.unique(y)
    log_prior = np.log(np.array([np.mean(y == c) for c in classes]))
    theta = np.array([(X[y == c].sum(0) + alpha) / ((y == c).sum() + 2 * alpha) for c in classes])
    return classes, log_prior, theta

def nb_predict_proba(model, X):
    classes, log_prior, theta = model
    L = log_prior + X @ np.log(theta).T + (1 - X) @ np.log(1 - theta).T
    L -= L.max(1, keepdims=True)
    P = np.exp(L)
    return P / P.sum(1, keepdims=True)

nb_model = nb_fit(X_words, y_words)
proba_nb = nb_predict_proba(nb_model, X_words)
```

</details>

### Exercise 12 · k-nearest neighbours from scratch
*💻 Code · ★☆☆*

Implement `knn_predict(X_train, y_train, X_test, k)` with Euclidean distance and majority vote. Compare with `KNeighborsClassifier`
on the iris split below for $k = 5$.

In [ ]:
X_iris, y_iris = load_iris(return_X_y=True)
Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(X_iris, y_iris, test_size=0.3, random_state=0)

def knn_predict(X_train, y_train, X_test, k=5):
    # TODO
    return None

pred_knn = knn_predict(Xtr_i, ytr_i, Xte_i, k=5)

check('same predictions as sklearn', pred_knn, KNeighborsClassifier(n_neighbors=5).fit(Xtr_i, ytr_i).predict(Xte_i))

<details>
<summary><b>💡 Hint</b></summary>

All pairwise distances: `np.linalg.norm(X_test[:, None, :] - X_train[None, :, :], axis=2)`; then `np.argsort(...)[:, :k]` and `np.bincount` per row.

</details>

<details>
<summary><b>✅ Solution</b></summary>

k-NN has no training phase (lazy learning): all the work happens at prediction time. Its inductive bias is that nearby points share labels,
which is why feature scaling (Lecture 08) matters so much.

```python
X_iris, y_iris = load_iris(return_X_y=True)
Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(X_iris, y_iris, test_size=0.3, random_state=0)

def knn_predict(X_train, y_train, X_test, k=5):
    D = np.linalg.norm(X_test[:, None, :] - X_train[None, :, :], axis=2)
    nn = np.argsort(D, axis=1, kind='stable')[:, :k]
    return np.array([np.bincount(y_train[row]).argmax() for row in nn])

pred_knn = knn_predict(Xtr_i, ytr_i, Xte_i, k=5)
```

</details>

### Exercise 13 · k-fold cross-validation from scratch
*💻 Code · ★★☆*

Implement `kfold_indices(n, k, seed)` returning a list of `(train_idx, test_idx)` pairs (shuffle once, then cut into $k$ nearly equal folds),
and `cv_accuracy(make_model, X, y, folds)` returning the mean test accuracy. Use 5 folds on iris to choose the number of neighbours
$k_{NN} \in \{1, 3, 5, \dots, 25\}$. Store the CV scores in `cv_scores` (dict) and the winner in `best_k`.

In [ ]:
def kfold_indices(n, k, seed=0):
    # TODO
    return None

def cv_accuracy(make_model, X, y, folds):
    # TODO
    return None

folds_iris = kfold_indices(len(y_iris), 5, seed=0)
cv_scores = None
best_k = None

if folds_iris is not None:
    _all = np.sort(np.concatenate([te for _, te in folds_iris]))
    check('test folds partition the data', _all, np.arange(len(y_iris)))
    check('CV score for k=5 matches cross_val_score', cv_scores[5],
          cross_val_score(KNeighborsClassifier(5), X_iris, y_iris, cv=folds_iris).mean())
    check('best_k maximises the CV score', cv_scores[best_k], max(cv_scores.values()), tol=1e-9)
else:
    check('test folds partition the data', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`perm = np.random.default_rng(seed).permutation(n)`, then `np.array_split(perm, k)`; the training part of fold $i$ is everything else.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Every example is used for testing exactly once, so the CV estimate uses all data and has lower variance than one hold-out split.
Several $k$ values usually tie; prefer the larger $k$ (smoother model) among ties, in the spirit of Ockham's razor. Remember: once $k$ is
chosen with CV, you need an untouched test set to report an unbiased final accuracy.

```python
def kfold_indices(n, k, seed=0):
    perm = np.random.default_rng(seed).permutation(n)
    parts = np.array_split(perm, k)
    return [(np.concatenate([p for j, p in enumerate(parts) if j != i]), parts[i]) for i in range(k)]

def cv_accuracy(make_model, X, y, folds):
    accs = [make_model().fit(X[tr], y[tr]).score(X[te], y[te]) for tr, te in folds]
    return float(np.mean(accs))

folds_iris = kfold_indices(len(y_iris), 5, seed=0)
cv_scores = {k: cv_accuracy(lambda: KNeighborsClassifier(k), X_iris, y_iris, folds_iris) for k in range(1, 26, 2)}
best_k = max(cv_scores, key=lambda k: (round(cv_scores[k], 10), k))
print(cv_scores, best_k)
```

</details>

### Exercise 14 · ROC curve and AUC
*💻 Code · ★★★*

Fit `LogisticRegression(max_iter=5000)` on the breast-cancer training split below and get test probabilities `scores`.
Implement `roc_points(y, scores)` returning arrays `(fpr, tpr)` obtained by sweeping the threshold over all distinct scores (from high to low,
starting at (0, 0)), and `auc(fpr, tpr)` with the trapezoidal rule. Compare with `roc_auc_score`.

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X_bc, y_bc, test_size=0.4, random_state=0)
scores = LogisticRegression(max_iter=5000).fit(Xtr_b, ytr_b).predict_proba(Xte_b)[:, 1]

def roc_points(y, scores):
    # TODO
    return None

def auc(fpr, tpr):
    # TODO
    return None

pts = roc_points(yte_b, scores)
my_auc = None if pts is None else auc(*pts)

check('AUC', my_auc, roc_auc_score(yte_b, scores))

<details>
<summary><b>💡 Hint 1</b></summary>

For each threshold $\tau$ in `np.r_[np.inf, np.unique(scores)[::-1]]`, predict positive when `scores >= tau`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Trapezoid: `np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each threshold gives one (FPR, TPR) point; together they trace the trade-off from the note between catching positives and raising false alarms.
AUC is the probability that a random positive gets a higher score than a random negative (0.5 = random, 1 = perfect). Here it is about 0.99.

```python
X_bc, y_bc = load_breast_cancer(return_X_y=True)
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X_bc, y_bc, test_size=0.4, random_state=0)
scores = LogisticRegression(max_iter=5000).fit(Xtr_b, ytr_b).predict_proba(Xte_b)[:, 1]

def roc_points(y, scores):
    P, N = (y == 1).sum(), (y == 0).sum()
    fpr, tpr = [], []
    for tau in np.r_[np.inf, np.unique(scores)[::-1]]:
        pred = scores >= tau
        tpr.append((pred & (y == 1)).sum() / P)
        fpr.append((pred & (y == 0)).sum() / N)
    return np.array(fpr), np.array(tpr)

def auc(fpr, tpr):
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))

pts = roc_points(yte_b, scores)
my_auc = auc(*pts)
print(my_auc)
```

</details>

### Exercise 15 · Gaussian naive Bayes for continuous features
*💻 Code · ★★★*

For real-valued features, naive Bayes models each $P(x_i\mid c)$ as a normal distribution with a per-class mean and variance (maximum
likelihood, i.e. `var(ddof=0)`). Implement `gnb_fit` / `gnb_predict_proba` and compare with `GaussianNB(var_smoothing=0)` on iris.

In [ ]:
def gnb_fit(X, y):
    # TODO: return (classes, priors, means, variances)
    return None

def gnb_predict_proba(model, X):
    # TODO
    return None

g_model = gnb_fit(X_iris, y_iris)
proba_g = None if g_model is None else gnb_predict_proba(g_model, X_iris)

check('matches GaussianNB', proba_g, GaussianNB(var_smoothing=0).fit(X_iris, y_iris).predict_proba(X_iris))

<details>
<summary><b>💡 Hint</b></summary>

$\log\mathcal N(x;\mu,\sigma^2) = -\tfrac12\log(2\pi\sigma^2) - \frac{(x-\mu)^2}{2\sigma^2}$; sum over features, add $\log P(c)$, normalise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The structure is the same Bayesian network as for spam; only the CPTs change from tables to Gaussian densities. On iris it reaches 96 %
training accuracy with just 2 parameters per feature and class.

```python
def gnb_fit(X, y):
    classes = np.unique(y)
    priors = np.array([np.mean(y == c) for c in classes])
    means = np.array([X[y == c].mean(0) for c in classes])
    variances = np.array([X[y == c].var(0) for c in classes])
    return classes, priors, means, variances

def gnb_predict_proba(model, X):
    classes, priors, means, variances = model
    L = np.log(priors) + np.stack([
        (-0.5 * np.log(2 * np.pi * variances[c]) - (X - means[c]) ** 2 / (2 * variances[c])).sum(1)
        for c in range(len(classes))], axis=1)
    L -= L.max(1, keepdims=True)
    P = np.exp(L)
    return P / P.sum(1, keepdims=True)

g_model = gnb_fit(X_iris, y_iris)
proba_g = gnb_predict_proba(g_model, X_iris)
```

</details>

### Exercise 16 · Model selection with a validation set
*💻 Code · ★★☆*

Split `make_moons(n_samples=600, noise=0.35, random_state=1)` into train / validation / test (60/20/20, `random_state=0` twice).
For decision trees with `max_depth` $\in \{1,\dots,15\}$, select the depth with the best **validation** accuracy (prefer the shallower tree on ties)
and report its **test** accuracy. Why must the test set not be used for the selection?

In [ ]:
best_depth_val = None
test_acc_selected = None
val_accs = None   # list of 15 validation accuracies

if val_accs is not None:
    check('selected depth is the shallowest best on validation', best_depth_val, int(np.argmax(val_accs)) + 1)
    check('test accuracy is reasonable', test_acc_selected > 0.8, True)
else:
    check('selected depth is the shallowest best on validation', None, 1)

<details>
<summary><b>💡 Hint</b></summary>

`train_test_split` twice: first 60/40, then split the 40 % half/half.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The selected depth is moderate (deep trees overfit the noisy moons). If the test set were used to pick the depth, the reported test accuracy
would be optimistically biased because we chose the model that happened to do best on exactly those points.

```python
Xm, ym = make_moons(n_samples=600, noise=0.35, random_state=1)
Xtr, Xrest, ytr, yrest = train_test_split(Xm, ym, test_size=0.4, random_state=0)
Xva, Xte, yva, yte = train_test_split(Xrest, yrest, test_size=0.5, random_state=0)
val_accs = [DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr).score(Xva, yva) for d in range(1, 16)]
best_depth_val = int(np.argmax(val_accs)) + 1
test_acc_selected = DecisionTreeClassifier(max_depth=best_depth_val, random_state=0).fit(Xtr, ytr).score(Xte, yte)
print(np.round(val_accs, 3), best_depth_val, test_acc_selected)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Noter små](../Noter%20små.md).*